# DEFINIÇÕES E CONSTANTES

In [ ]:
# ---------- Paths ----------
PASTA_GERAL = "Dados/"
PASTA_DADOS = PASTA_GERAL + "Arquivos/"
SETUP_TESTES_XLSX = "Testes.xlsx"
NOME_ARQUIVO_LOG = "TEST_X.CSV"

# ---------- Pré-Processamento ----------
TEMPO_LIMPEZA_INICIAL = 5  # segundos
TIMESTAMP_PARA_SEGUNDOS = 1e3  # converter timestamp para segundos
NOME_VARIAVEL_TEMPO = "Timestamp"

# HELPERS

In [ ]:
def descritiva(df_, var, vresp='survived', max_classes=5):
    """
    Gera um gráfico descritivo da taxa de sobreviventes por categoria da variável especificada.
    
    Parâmetros:
    df : DataFrame - Base de dados a ser analisada.
    var : str - Nome da variável categórica a ser analisada.
    """
    
    df = df_.copy()
    
    if df[var].nunique()>max_classes:
        df[var] = pd.qcut(df[var], max_classes, duplicates='drop')
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    sns.pointplot(data=df, y=vresp, x=var, ax=ax1)
    
    # Criar o segundo eixo y para a taxa de sobreviventes
    ax2 = ax1.twinx()
    sns.countplot(data=df, x=var, palette='viridis', alpha=0.5, ax=ax2)
    ax2.set_ylabel('Frequência', color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    
    ax1.set_zorder(2)
    ax1.patch.set_visible(False)  # Tornar o fundo do eixo 1 transparente
    
    # Exibir o gráfico
    plt.show()
    
def relatorio_missing(df):
    print(f'Número de linhas: {df.shape[0]} | Número de colunas: {df.shape[1]}')
    return pd.DataFrame({'Pct_missing': df.isna().mean().apply(lambda x: f"{x:.1%}"),
                          'Freq_missing': df.isna().sum().apply(lambda x: f"{x:,.0f}").replace(',','.')})
    

# Importações de Biblioteca

In [ ]:
import os
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# from helpers import descritiva

# Carregar Arquivos

## Carregar Testes.xlsx

In [ ]:
# ---------- 1) Ler metadados do Excel ----------
df_meta = pd.read_excel(os.path.join(PASTA_GERAL, "Testes.xlsx"))

# Normaliza nomes de colunas (caso venham com espaços/variações)
df_meta = df_meta.rename(columns={
    "ID TEST": "ID_TEST",
})

# Garante tipos
df_meta["ID_TEST"] = df_meta["ID_TEST"].astype(int)

# Cria um dicionário: id -> (vel, desb)
meta_map = (
    df_meta
    .set_index("ID_TEST")[["Velocity", "Imbalance"]]
    .to_dict(orient="index")
)



## Carregar os Dados

In [ ]:
dados_original_separados = []  # Lista para armazenar os dados lidos
dados_original_juntos = []     # Lista para armazenar os dados combinados

# ---------- 1) Definindo local do arquivo ----------
padrao = re.compile(NOME_ARQUIVO_LOG.replace("X", r"(\d+)"), re.IGNORECASE)

# ---------- 2) Lendo arquivos CSV ----------
for nome_arquivo in os.listdir(PASTA_DADOS):
    correspondencia = padrao.match(nome_arquivo)
    if correspondencia:
        id_teste = int(correspondencia.group(1))
        caminho_arquivo = os.path.join(PASTA_DADOS, nome_arquivo)
        
        # Lê o arquivo CSV
        df_dados = pd.read_csv(caminho_arquivo)
        
        # Adiciona colunas de metadados
        if id_teste in meta_map:
            df_dados["Velocity"] = meta_map[id_teste]["Velocity"]
            df_dados["Imbalance"] = meta_map[id_teste]["Imbalance"]
            df_dados["ID_TEST"] = id_teste
            
            dados_original_separados.append(df_dados)
            
            print(f"Lido arquivo: {nome_arquivo} com ID_TEST {id_teste}")
        else:
            print(f"Metadados não encontrados para ID_TEST {id_teste}")
            
# ---------- 3) Junta tudo em um único DataFrame ----------
dados_original_juntos = pd.concat(dados_original_separados, ignore_index=True)

### Checagem rápida dos dados

In [ ]:
print(dados_original_juntos.head())
print(dados_original_juntos.columns)

# Pre Processing

## Limpeza inicial dos dados

In [ ]:
# Limpar os primeiros segundos de cada teste carregado
for id_teste in dados_original_juntos["ID_TEST"].unique():
    filtro_teste = dados_original_juntos["ID_TEST"] == id_teste
    limite_tempo = (TEMPO_LIMPEZA_INICIAL) * TIMESTAMP_PARA_SEGUNDOS
    
    dados_original_juntos = dados_original_juntos[~(filtro_teste & (dados_original_juntos[NOME_VARIAVEL_TEMPO] < limite_tempo))]



# Análise Geral

## Verifica possíveis missing

In [20]:
relatorio_missing(dados_original_juntos)

Número de linhas: 135000 | Número de colunas: 13


,Pct_missing,Freq_missing
Timestamp,0.0%,0
Accel_X,0.0%,0
Accel_Y,0.0%,0
Accel_Z,0.0%,0
Gyro_X,0.0%,0
Gyro_Y,0.0%,0
Gyro_Z,0.0%,0
Mag_X,0.0%,0
Mag_Y,0.0%,0
Mag_Z,0.0%,0


## Verifica os tipos

In [21]:
metadados = dados_original_juntos.dtypes
print(metadados)

Timestamp    int64
Accel_X      int64
Accel_Y      int64
Accel_Z      int64
Gyro_X       int64
Gyro_Y       int64
Gyro_Z       int64
Mag_X        int64
Mag_Y        int64
Mag_Z        int64
Velocity     int64
Imbalance    int64
ID_TEST      int64
dtype: object


## Análise descritiva

In [ ]:
# Análise descritiva básica

for variavel in dados_original_juntos.columns:
    print(f'\n\nAnálise univariada de {variavel}:')
    print(dados_original_juntos[variavel].describe())
    
    descritiva(dados_original_juntos, variavel, vresp='Imbalance', max_classes=5)